# Cookie Cats EDA

In [ ]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')


## Cookie Cats A/B Test — Exploratory Data Analysis

[Cookie Cats](https://www.kaggle.com/datasets/meirnizri/cookie-cats) is a popular mobile puzzle game.
The A/B test moved the first "gate" (a forced wait before continuing) from level 30 to level 40.
The key metrics are **1-day retention** and **7-day retention**.

> **Data required:** place the CSV at `data/raw/cookie_cats.csv`.
> See `data/README.md` for the one-step Kaggle download command.


In [ ]:
import pandas as pd
from pathlib import Path

csv = Path('data/raw/cookie_cats.csv')
if not csv.exists():
    print('cookie_cats.csv not found — see data/README.md for the one-step download. Skipping live cells.')
    df = None
else:
    df = pd.read_csv(csv)
    df.head()


### Group sizes & retention rates


In [ ]:
if df is not None:
    print("Group sizes:")
    print(df['version'].value_counts())
    print()
    print("Retention means by version:")
    print(df.groupby('version')[['retention_1', 'retention_7']].mean().round(4))


### Retention bar chart


In [ ]:
if df is not None:
    import matplotlib.pyplot as plt

    ret = df.groupby('version')[['retention_1', 'retention_7']].mean()
    ax = ret.plot(kind='bar', figsize=(7, 4), rot=0,
                  color=['#4C72B0', '#DD8452'])
    ax.set_title('Cookie Cats — Retention by Gate Version')
    ax.set_xlabel('Version')
    ax.set_ylabel('Retention Rate')
    ax.legend(['1-day', '7-day'])
    plt.tight_layout()
    plt.show()


### Sample Ratio Mismatch (SRM) check on 7-day retention


In [ ]:
if df is not None:
    from src.abtest import ExperimentData, srm_check

    df2 = df.rename(columns={'version': 'variant', 'retention_7': 'converted'})
    df2['variant'] = df2['variant'].map({'gate_30': 'control', 'gate_40': 'treatment'})
    data = ExperimentData(
        df2[['variant', 'converted']].assign(unit_id=range(len(df2))),
        metric_cols=['converted'],
    )
    result = srm_check(data)
    print(f"SRM check passed: {result.passed}")
    print(f"chi2 statistic : {result.statistic:.4f}")
    print(f"p-value        : {result.p_value:.4f}")
    print(f"Detail         : {result.detail}")
